# Access the grib file

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import cfgrib
import os
import glob

In [3]:
grib_data=cfgrib.open_datasets('/scratch/b5at/ranil.b5at/met_raw/era5monthly_global/data.grib')
grib_data # Tehre is a list of datasets based on whetherr grid and time is exact. Here we have two datasets as theres a 12hr offset in the data for t2m and 10mwindspeed vs tp and ssrd

/home/b5at/ranil.b5at/miniforge3/envs/hydra_bnf_env/lib/python3.12/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/home/b5at/ranil.b5at/miniforge3/envs/hydra_bnf_env/lib/python3.12/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly

[<xarray.Dataset> Size: 9GB
 Dimensions:     (time: 1033, latitude: 721, longitude: 1440)
 Coordinates:
   * time        (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2026-01-01
   * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
   * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
     number      int64 8B 0
     step        timedelta64[ns] 8B 00:00:00
     surface     float64 8B 0.0
     valid_time  (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2026-01-01
 Data variables:
     t2m         (time, latitude, longitude) float32 4GB ...
     si10        (time, latitude, longitude) float32 4GB ...
 Attributes:
     GRIB_edition:            1
     GRIB_centre:             ecmf
     GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
     GRIB_subCentre:          0
     Conventions:             CF-1.7
     institution:             European Centre for Medium-Range Weather Forecasts,
 <xarray.Dataset> S

In [4]:
ds_inst = grib_data[0]   # t2m, si10
ds_acc  = grib_data[1]   # ssrd, tp
ds0 = grib_data[0].swap_dims({"time": "valid_time"}).drop_vars("time").rename({"valid_time": "time"})
ds1 = grib_data[1].swap_dims({"time": "valid_time"}).drop_vars("time").rename({"valid_time": "time"})

ds = xr.merge([ds0, ds1], compat="override")

/local/user/1483801751/ipykernel_140327/2692776204.py:6: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  ds = xr.merge([ds0, ds1], compat="override")


In [8]:
ds

<xarray.Dataset> Size: 34GB
Dimensions:    (latitude: 721, longitude: 1440, time: 2066)
Coordinates:
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time       (time) datetime64[ns] 17kB 1940-01-01 ... 2026-01-01T06:00:00
    number     int64 8B 0
    step       timedelta64[ns] 8B 00:00:00
    surface    float64 8B 0.0
Data variables:
    t2m        (time, latitude, longitude) float32 9GB 247.6 247.6 ... nan nan
    si10       (time, latitude, longitude) float32 9GB 6.114 6.114 ... nan nan
    ssrd       (time, latitude, longitude) float32 9GB nan nan ... 3.319e+07
    tp         (time, latitude, longitude) float32 9GB nan nan ... 0.0001593
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

In [5]:
ds = ds.chunk({"time": 12})

In [6]:
ds

<xarray.Dataset> Size: 34GB
Dimensions:    (latitude: 721, longitude: 1440, time: 2066)
Coordinates:
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time       (time) datetime64[ns] 17kB 1940-01-01 ... 2026-01-01T06:00:00
    number     int64 8B 0
    step       timedelta64[ns] 8B 00:00:00
    surface    float64 8B 0.0
Data variables:
    t2m        (time, latitude, longitude) float32 9GB dask.array<chunksize=(12, 721, 1440), meta=np.ndarray>
    si10       (time, latitude, longitude) float32 9GB dask.array<chunksize=(12, 721, 1440), meta=np.ndarray>
    ssrd       (time, latitude, longitude) float32 9GB dask.array<chunksize=(12, 721, 1440), meta=np.ndarray>
    tp         (time, latitude, longitude) float32 9GB dask.array<chunksize=(12, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

# Generate annual aggregated nc files for sampling

In [10]:
t2m_ann = xr.Dataset({
    "t2m_mean":   ds["t2m"].resample(time="YS").mean(skipna=True),
    "t2m_median": ds["t2m"].resample(time="YS").median(skipna=True),
    "t2m_std":    ds["t2m"].resample(time="YS").std(skipna=True),
    "t2m_min":    ds["t2m"].resample(time="YS").min(skipna=True),
    "t2m_max":    ds["t2m"].resample(time="YS").max(skipna=True),
})

# ---------- tp annual stats (monthly accumulations -> annual sum) ----------
tp_ann = xr.Dataset({
    "tp_sum": ds["tp"].resample(time="YS").sum(skipna=True),
    "tp_std": ds["tp"].resample(time="YS").std(skipna=True),
})

# ---------- ssrd annual stats (monthly accumulations -> annual sum) ----------
ssrd_ann = xr.Dataset({
    "ssrd_sum": ds["ssrd"].resample(time="YS").sum(skipna=True),
    "ssrd_std": ds["ssrd"].resample(time="YS").std(skipna=True),
})

# Combine
ds_annual = xr.merge([t2m_ann, tp_ann, ssrd_ann], compat="override")

ds_annual

<xarray.Dataset> Size: 3GB
Dimensions:     (latitude: 721, longitude: 1440, time: 87)
Coordinates:
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time        (time) datetime64[ns] 696B 1940-01-01 1941-01-01 ... 2026-01-01
    number      int64 8B 0
    step        timedelta64[ns] 8B 00:00:00
    surface     float64 8B 0.0
Data variables:
    t2m_mean    (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_median  (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_std     (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_min     (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_max     (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp_sum      (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp_std      (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ssrd_sum    (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ssrd_std    (time, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>

In [11]:
ds_annual.to_netcdf("/scratch/b5at/ranil.b5at/met_raw/era5aggregates/era5_annual_aggregates.nc")

# Generate seasonal aggregated nc files for sampling

In [6]:
# unit conversions from era5 defaults
ds["tp"]   = ds["tp"] * 1000.0   # m -> mm
ds["ssrd"] = ds["ssrd"] / 1e6    # J/m2 -> MJ/m2
ds["t2m"]  = ds["t2m"] - 273.15  # K -> C

def season_year(time, start_month):
    # Here we assume that year represents when the glacier is at its minimum mass (ie peak ablation). So start the accumulation from the previous cycle until we reach it
    # eg for the year 2000 we mean:
    # nh_acc: 1999 Oct–Dec + 2000 Jan–Ap
    # nh_abl: 2000 May–Sep where min mass fo glacier is at the end of Sep 2000. Above we picked the acc season preceding.
    # sh_acc: 1999 Apr-Oct
    # sh_abl: 1999 Nov-Dec and 2000 Jan-Mar where minimum mass of glacier is at end of March 2000. Above we picked the acc season preceding.

    # label by the year in which the season-year ends
    # e.g., NH start_month=10 => Oct/Nov/Dec 2000 -> season_year 2001
    y = time.dt.year
    m = time.dt.month
    return y + (m >= start_month) 

def agg_months(ds_in, months, label, start_month):
    sub = ds_in.where(ds_in.time.dt.month.isin(months), drop=True) # Masks only for the specified set of months
    sub = sub.assign_coords(season_year=("time", season_year(sub.time, start_month).values)) #Creates season year coordinate to allow grouping and aggregations

    return xr.Dataset({
        f"tp_{label}_sum":   sub["tp"].groupby("season_year").sum("time"),
        f"tp_{label}_std":   sub["tp"].groupby("season_year").std("time"),
        f"ssrd_{label}_sum": sub["ssrd"].groupby("season_year").sum("time"),
        f"ssrd_{label}_std": sub["ssrd"].groupby("season_year").std("time"),
        f"t2m_{label}_mean": sub["t2m"].groupby("season_year").mean("time"),
        f"t2m_{label}_std":  sub["t2m"].groupby("season_year").std("time"),
    })

# month sets
nh_acc = [10, 11, 12, 1, 2, 3, 4]   # Oct–Apr
nh_abl = [5, 6, 7, 8, 9]            # May–Sep
sh_acc = [4, 5, 6, 7, 8, 9, 10]     # Apr–Oct
sh_abl = [11, 12, 1, 2, 3]          # Nov–Mar

# water-year starts
NH_START = 10  # Oct or alternately if nh_acc is ordered nh_acc[0]
SH_START = 4   # Apr

nh = xr.merge([
    agg_months(ds, nh_acc, "accum", NH_START),
    agg_months(ds, nh_abl, "ablat", NH_START),
])

sh = xr.merge([
    agg_months(ds, sh_acc, "accum", SH_START),
    agg_months(ds, sh_abl, "ablat", SH_START),
])

/local/user/1483801751/ipykernel_140327/1385964620.py:43: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'season_year' ('season_year',) The recommendation is to set join explicitly for this case.
  nh = xr.merge([
/local/user/1483801751/ipykernel_140327/1385964620.py:43: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  nh = xr.merge([
/local/user/1483801751/ipykernel_140327/1385964620.py:43: FutureWarning: In a future version o

In [ ]:
# combine by latitude
mask_nh = ds["latitude"] >= 0
features = nh.where(mask_nh).fillna(sh.where(~mask_nh)).sortby("season_year") # the where masks the latitudes for nh and fills the rest with na. fillna replaces the na values with the sh values where mask is opposite. sortby is optional

# write
out_nc = "/scratch/b5at/ranil.b5at/met_raw/era5aggregates/era5_seasonal_accum_ablat.nc"
encoding = {v: {"zlib": True, "complevel": 4, "dtype": "float32"} for v in features.data_vars}
features.to_netcdf(out_nc, encoding=encoding)
print(out_nc)

/scratch/b5at/ranil.b5at/met_raw/era5monthly_global/era5_seasonal_accum_ablat.nc


In [8]:
features

<xarray.Dataset> Size: 4GB
Dimensions:         (season_year: 87, latitude: 721, longitude: 1440)
Coordinates:
  * season_year     (season_year) int64 696B 1940 1941 1942 ... 2024 2025 2026
  * latitude        (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude       (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    number          int64 8B 0
    step            timedelta64[ns] 8B 00:00:00
    surface         float64 8B 0.0
Data variables:
    tp_accum_sum    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp_accum_std    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ssrd_accum_sum  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ssrd_accum_std  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_accum_mean  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_accum_std   (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp_ablat_sum    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp_ablat_std    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ssrd_ablat_sum  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ssrd_ablat_std  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_ablat_mean  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m_ablat_std   (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>

## Sample glaciers from seasonal

In [2]:
# Read csv file ocntaining mass balance targets with lat, lon, time/year columns
gla_file=pd.read_csv('/scratch/b5at/ranil.b5at/met_raw/era5aggregates/iceland_fog_mass_balance.csv')

gla_file.head(2)

,glacier_id,glacier_name,year,annual_balance,id,latitude,longitude,rgi50_ids,rgi60_ids
0,3067,BRUARJOKULL,1993,1.09,3067,64.669998,-16.17,RGI50-06.00377,RGI60-06.00377
1,3067,BRUARJOKULL,1994,0.55,3067,64.669998,-16.17,RGI50-06.00377,RGI60-06.00377


In [7]:
clim_file=xr.open_dataset('/scratch/b5at/ranil.b5at/met_raw/era5aggregates/era5_seasonal_accum_ablat.nc',chunks={"season_year": 10})
clim_file

/local/user/1483801751/ipykernel_270792/2832448674.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "season_year" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  clim_file=xr.open_dataset('/scratch/b5at/ranil.b5at/met_raw/era5aggregates/era5_seasonal_accum_ablat.nc',chunks={"season_year": 10})


<xarray.Dataset> Size: 4GB
Dimensions:         (season_year: 87, latitude: 721, longitude: 1440)
Coordinates:
  * season_year     (season_year) int64 696B 1940 1941 1942 ... 2024 2025 2026
  * latitude        (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude       (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    number          int64 8B ...
    step            timedelta64[ns] 8B ...
    surface         float64 8B ...
Data variables:
    tp_accum_sum    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    tp_accum_std    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    ssrd_accum_sum  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    ssrd_accum_std  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    t2m_accum_mean  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    t2m_accum_std   (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    tp_ablat_sum    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    tp_ablat_std    (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    ssrd_ablat_sum  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    ssrd_ablat_std  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    t2m_ablat_mean  (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>
    t2m_ablat_std   (season_year, latitude, longitude) float32 361MB dask.array<chunksize=(10, 241, 480), meta=np.ndarray>

In [8]:
clim_file.longitude.max()

<xarray.DataArray 'longitude' ()> Size: 8B
array(359.75)
Coordinates:
    number   int64 8B ...
    step     timedelta64[ns] 8B ...
    surface  float64 8B ...
Attributes:
    units:          degrees_east
    standard_name:  longitude
    long_name:      longitude

In [19]:
# --- Convert glacier longitudes to 0–360 ---
gla_lons = (gla_file["longitude"].to_numpy() + 360) % 360
gla_lats = gla_file["latitude"].to_numpy()
gla_years = gla_file["year"].to_numpy()

# --- Create index dimension for glaciers for merging with csv later---
glacier_ids = gla_file["glacier_id"].to_numpy()

lats = xr.DataArray(gla_lats, dims="glacier")
lons = xr.DataArray(gla_lons, dims="glacier")
yrs = xr.DataArray(gla_years,dims="glacier")

# --- Nearest neighbour sampling ---
sampled = clim_file.sel(
    latitude=lats,
    longitude=lons,
    season_year=yrs,
    method="nearest"
)

# Add glacier_id as coordinate
sampled = sampled.assign_coords(glacier=("glacier", glacier_ids))

# --- Convert to dataframe ---
df = sampled.to_dataframe().reset_index()

# Optional: merge back glacier metadata if needed
# df = df.merge(gla_file[["glacier_id", "glacier_name"]], on="glacier_id", how="left")

In [24]:
df.head(2)

,glacier,tp_accum_sum,tp_accum_std,ssrd_accum_sum,ssrd_accum_std,t2m_accum_mean,t2m_accum_std,tp_ablat_sum,tp_ablat_std,ssrd_ablat_sum,ssrd_ablat_std,t2m_ablat_mean,t2m_ablat_std,latitude,longitude,season_year,number,step,surface
0,3067,32.252312,1.698414,29.524773,4.683159,-4.059745,2.127895,15.821457,1.074352,68.649681,5.041136,3.165356,1.898222,64.75,343.75,1993,0,0 days,0.0
1,3067,34.942627,2.708240,30.936638,5.180821,-4.509731,2.644110,12.199402,0.753027,82.283226,4.951058,4.197003,3.120204,64.75,343.75,1994,0,0 days,0.0


In [25]:
gla_file.head(2)

,glacier_id,glacier_name,year,annual_balance,id,latitude,longitude,rgi50_ids,rgi60_ids
0,3067,BRUARJOKULL,1993,1.09,3067,64.669998,-16.17,RGI50-06.00377,RGI60-06.00377
1,3067,BRUARJOKULL,1994,0.55,3067,64.669998,-16.17,RGI50-06.00377,RGI60-06.00377


In [26]:
df.rename(columns={"season_year": "year", "glacier": "glacier_id"}, inplace=True)
df.head(2)

,glacier_id,tp_accum_sum,tp_accum_std,ssrd_accum_sum,ssrd_accum_std,t2m_accum_mean,t2m_accum_std,tp_ablat_sum,tp_ablat_std,ssrd_ablat_sum,ssrd_ablat_std,t2m_ablat_mean,t2m_ablat_std,latitude,longitude,year,number,step,surface
0,3067,32.252312,1.698414,29.524773,4.683159,-4.059745,2.127895,15.821457,1.074352,68.649681,5.041136,3.165356,1.898222,64.75,343.75,1993,0,0 days,0.0
1,3067,34.942627,2.708240,30.936638,5.180821,-4.509731,2.644110,12.199402,0.753027,82.283226,4.951058,4.197003,3.120204,64.75,343.75,1994,0,0 days,0.0


In [27]:
fin_merged_df = df.merge(gla_file[["glacier_id", "glacier_name", "annual_balance","year"]], on=["glacier_id","year"], how="inner")

In [28]:
fin_merged_df.shape

(281, 21)

In [29]:
fin_merged_df.columns

Index(['glacier_id', 'tp_accum_sum', 'tp_accum_std', 'ssrd_accum_sum',
       'ssrd_accum_std', 't2m_accum_mean', 't2m_accum_std', 'tp_ablat_sum',
       'tp_ablat_std', 'ssrd_ablat_sum', 'ssrd_ablat_std', 't2m_ablat_mean',
       't2m_ablat_std', 'latitude', 'longitude', 'year', 'number', 'step',
       'surface', 'glacier_name', 'annual_balance'],
      dtype='object')

In [30]:
fin_merged_df.to_csv('/scratch/b5at/ranil.b5at/met_raw/era5aggregates/iceland_era5_sampled.csv')